# Citadel Kaggle TPU launcher (thin — secondary surface)
All logic lives in `citadel_tpu/`. Flow: repo → pinned Cymek runtime → preflight gate → probe → T0 → STOP unless T0 passes. Branch: `citadel`. Cymek is never merged; its SHA is recorded per receipt. No secrets are used or printed.

In [ ]:
# 0. Repo: use the pushed citadel branch (uploaded worktree or clone); record where we are
!git rev-parse HEAD 2>/dev/null || echo NO_GIT
!git log -1 --oneline 2>/dev/null || true

In [ ]:
# 1. Pinned read-only Cymek runtime (stdlib only; no device use). Prints both codebase identities.
import os
os.environ['CITADEL_PLATFORM'] = 'kaggle'
from citadel_tpu import runtime_bootstrap as rb
rt_root, rt_sha = rb.ensure_cymek_runtime()
print('CITADEL_SHA=' + str(rb.citadel_sha()))
print('CYMEK_RUNTIME_SHA=' + str(rt_sha))
print('CYMEK_RUNTIME_PATH=' + str(rt_root))

In [ ]:
# 2. Inspect runtime first; install ONLY what is missing (Kaggle preinstalls its TPU stack — do not overwrite it)
!python -c "import torch; print('torch', torch.__version__)" 2>&1 | tail -1
!python -c "import torch_xla; print('torch-xla', getattr(torch_xla, '__version__', 'unknown'))" 2>&1 | tail -1
# !pip install -q numpy 2>&1 | tail -1  (only if the inspection above shows it missing)

In [ ]:
# 3. Preflight gate: every import, file, API and the TPU itself verified. If this fails, do NOT run T0.
import subprocess
r = subprocess.run(['python','-m','citadel_tpu.preflight'])
assert r.returncode == 0, 'PREFLIGHT failed — READY_FOR_T0=NO. Diagnose, do not escalate.'
print('PREFLIGHT READY_FOR_T0=YES')

In [ ]:
# 4. M0: environment probe (fail-closed; ABORT_NO_TPU on CPU fallback)
from citadel_tpu import environment as env_mod
env = env_mod.main(out='docs/citadel/tpu_receipts/TPU_ENVIRONMENT.json', require_tpu=True, platform_override='kaggle')
print(env)

In [ ]:
# 5. T0: single-device one-update certification (MINI_SPEC, bucket 512)
from citadel_tpu import one_update
r0 = one_update.run(out='docs/citadel/tpu_receipts/TPU_ONE_UPDATE.json')
print({k: r0[k] for k in ('citadel_sha','cymek_runtime_sha','certification','loss','tokens_per_second','reload_identical')})

In [ ]:
# 6. STOP unless T0 passed. Canary data is deterministic and overlap-guarded.
assert r0.get('certification') == 'PASS', 'T0 did not pass — STOP. Diagnose, do not escalate.'
from citadel_tpu import calculator_data as calc
rc = calc.build_all(out_dir='docs/citadel/tpu_receipts/calculator_canary')
print(rc['splits'], rc['split_overlap_rows'])

In [ ]:
# 7. T1: calculator checkpoint (CE only; infrastructure gate, not AGI)
from citadel_tpu import calculator_train
r1 = calculator_train.train(out='docs/citadel/tpu_receipts/TPU_CALCULATOR_CHECKPOINT.json')
print(r1['training'], r1['eval'], r1['interpretation'])

In [ ]:
# 8. Throughput (cold vs steady; steady sizes the 5B plan) + multi-device ledger check
from citadel_tpu import throughput as tp
print(tp.measure_steady_state())
print(tp.certify_multi_device())